<a href="https://colab.research.google.com/github/cras-lab/LangChain/blob/main/Agent_AI_with_Structured_DB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

모델선택

In [1]:
LLM_PROVIDER = "openai" # claude 또는 gemini
KEYFROM_USERDATA = True  # 외부에서 입력하려면 False 로
TEMPERATURE = 0.2

필요한 모듈을 설치한다.

In [2]:
pip install -q -U langchain langchain-openai langchain-anthropic langchain-google-genai

먼저 키부터 입력 받는다.

In [16]:
if KEYFROM_USERDATA:
  from google.colab import userdata
  API_KEY = userdata.get('OPENAI').strip()
else:
  import getpass
  API_KEY = getpass.getpass('Enter API key: ')

정형 DB로는 sqlite3를 사용하기로 한다. 이를 임포트

In [17]:
import sqlite3

Langchain 관련 모듈을 임포트 한다.

In [18]:
from langchain.tools import tool
from langchain.agents import create_agent

conn은 데이터베이스 연결 객체<BR>
cursor는 연결 위에서 SQL 명령실행하는 작업 핸들<BR>
":memory:" → 파일 저장(X), 메모리 안에만 임시 DB

In [19]:
conn = sqlite3.connect(":memory:", check_same_thread=False)
cursor = conn.cursor()  # 실질적으로 실행. 현재 DB위치

TV용 테이블 설정

In [ ]:
cursor.execute("""
CREATE TABLE tv_products (
    id TEXT,
    name TEXT,
    brand TEXT,
    size INTEGER,
    year INTEGER,
    price INTEGER,
    panel TEXT
)
""")

임의의 LG TV를 구성

In [21]:
data = [
    ("tv001", "LG UHD AI TV 50UA9000", "LG", 60, 2026, 690000, "LED"),
    ("tv002", "LG QNED TV 50QNED85A", "LG", 50, 2026, 890000, "QNED"),
    ("tv003", "LG UHD AI TV 50UA8500", "LG", 80, 2025, 1500000, "LED"),
    ("tv004", "LG NanoCell 50NANO80", "LG", 50, 2025, 770000, "NanoCell"),
    ("tv005", "LG UHD TV 55UA9000", "LG", 60, 2026, 790000, "LED"),
    ("tv006", "Samsung Crystal UHD 50CU8000", "Samsung", 50, 2025, 640000, "LED"),
    ("tv007", "LG OLED evo 48C6", "LG", 48, 2026, 1490000, "OLED"),
    ("tv008", "LG UHD AI TV 50UA7000", "LG", 50, 2024, 540000, "LED"),
    ("tv009", "Sony Bravia 50X80K", "Sony", 50, 2025, 830000, "LED"),
    ("tv010", "LG QNED TV 50QNED80", "LG", 30, 2024, 710000, "QNED"),
]

해당 리시트를 DB에 실제로 저장

In [22]:
cursor.executemany("INSERT INTO tv_products VALUES (?, ?, ?, ?, ?, ?, ?)", data)
conn.commit()

Agent 툴 정의: TV로 상품조회하는 툴 정의

In [23]:
@tool
def query_tv_database(sql: str) -> str:
    """
    SQLite 데이터베이스에 SQL을 실행해서 TV 상품 정보를 조회한다.
    SELECT 문만 허용한다.
    """
    try:
        sql_clean = sql.strip().lower()
        if not sql_clean.startswith("select"):
            return "오류: SELECT 쿼리만 허용됩니다."

        rows = cursor.execute(sql).fetchall()
        columns = [desc[0] for desc in cursor.description] if cursor.description else []

        if not rows:
            return "조회 결과가 없습니다."

        results = [dict(zip(columns, row)) for row in rows]
        return str(results)

    except Exception as e:
        return f"SQL 실행 오류: {e}"

입력을 처리하는 챗봇을 생성한다.

In [ ]:
if LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(
        model="gpt-5-nano",
        temperature=TEMPERATURE,
        api_key=API_KEY,
      )

elif LLM_PROVIDER == "claude":
    from langchain_anthropic import ChatAnthropic

    llm = ChatAnthropic(
        model="claude-sonnet-4-6",
        temperature=TEMPERATURE,
        api_key=API_KEY,
      )


elif LLM_PROVIDER == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=TEMPERATURE,
        api_key=API_KEY,
      )


Agent를 생성한다. Agent에게 역할과 초기 지침을 준다.

In [25]:
agent = create_agent(
    model=llm,
    tools=[query_tv_database],
    system_prompt="""
너는 TV 추천 AI 에이전트다.
사용자의 자연어 요청을 분석해서 적절한 SQL을 만들고, 반드시 query_tv_database tool을 호출하라.

DB 테이블 이름은 tv_products 이다.
컬럼은 다음과 같다:
- id
- name
- brand
- size
- year
- price
- panel

규칙:
- 'LG'는 brand = 'LG'
- '삼성' 또는 'Samsung'은 brand = 'Samsung'
- '소니' 또는 'Sony'는 brand = 'Sony'
- '50인치'는 size = 50
- '55인치'는 size = 55
- '최신'은 ORDER BY year DESC
- '저렴' 또는 '가장 싼'은 ORDER BY price ASC
- '최신이며 저렴'은 ORDER BY year DESC, price ASC
- 추천 1개면 LIMIT 1 사용
- 반드시 SQLite 문법에 맞게 SQL을 작성하라
- SQL은 SELECT만 사용하라

최종 답변은:
1. 검색 조건 요약
2. 추천 제품
3. 연도, 가격, 패널 정보
순서로 간단히 정리하라.
"""
)

채팅을 이어나갈 함수를 정의한다.

In [26]:
# ---------------------------------
# 채팅 루프
# ---------------------------------
def chat_loop():
    print("TV 추천 AI 에이전트입니다.")
    print("종료하려면 'exit' 또는 'quit'를 입력하세요.\n")

    while True:
        user_input = input("사용자> ").strip()

        if user_input.lower() in ["exit", "quit"]:
            print("종료합니다.")
            break

        if not user_input:
            continue

        try:
            result = agent.invoke({
                "messages": [
                    {"role": "user", "content": user_input}
                ]
            })
            print("\n에이전트>")
            print(result["messages"][-1].content)
            print()

        except Exception as e:
            print(f"\n에러 발생: {e}\n")


채팅을 시작한다.

In [ ]:
chat_loop()